# 04 · Train a Bradley-Terry reward model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/04_reward_model.ipynb) &nbsp; 📖 Concept deck: [Reward models](https://nilesh-salpe.github.io/visual-ai/post-training/04-reward-models/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** the pairwise preference loss used by InstructGPT and Llama 2, a reward model trained on simulated human comparisons, and the conversion from K-way rankings to training pairs.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

rng = np.random.default_rng(0)
d = 16
w_true = rng.normal(size=d)                      # the hidden "human" reward direction
def make_pairs(n, seed):
    """Two responses per prompt (feature vectors). The human prefers A with prob sigmoid(r_A - r_B) (Bradley-Terry)."""
    r_ = np.random.default_rng(seed)
    A, Bv = r_.normal(size=(n, d)), r_.normal(size=(n, d))
    pA = 1 / (1 + np.exp(-(A @ w_true - Bv @ w_true) / 2))
    a_wins = r_.random(n) < pA
    win = np.where(a_wins[:, None], A, Bv); lose = np.where(a_wins[:, None], Bv, A)
    return win, lose
train_w, train_l = make_pairs(2000, 1)
test_w, test_l = make_pairs(2000, 2)

## Task 1: The pairwise loss

A reward model $r_\theta(x, y)$ outputs one scalar per response. Under the **Bradley-Terry** model, the probability that $y_w$ beats $y_l$ is $\sigma(r_w - r_l)$. Train by maximizing the log-likelihood of the observed choices (InstructGPT eq. 1):
$$\mathcal{L} = -\mathbb{E}\big[\log \sigma(r_\theta(x,y_w) - r_\theta(x,y_l))\big]$$
Use `log_sigmoid` (numerically stable). Only **differences** matter, so adding a constant to every reward changes nothing.

In [ ]:
def bt_loss(r_w, r_l):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_bt_loss(f):
    assert np.isclose(f(np.array([0.0]), np.array([0.0])), np.log(2)), "equal rewards -> loss log 2"
    rw, rl = np.array([2.0, -1.0, 50.0]), np.array([1.0, 1.0, -50.0])
    assert np.isclose(f(rw, rl), np.mean([np.log1p(np.exp(-1)), np.log1p(np.exp(2)), 0.0]), atol=1e-12)
    assert np.isclose(f(rw + 7, rl + 7), f(rw, rl)), "shift invariance"
    assert np.isfinite(f(np.array([-1000.0]), np.array([1000.0]))), "must be numerically stable"

check("bt_loss", _test_bt_loss, bt_loss)

## Task 2: Fit a linear reward model

Our "responses" are feature vectors, and the reward model is linear: $r = \phi \cdot w$. Fit $w$ by gradient descent on the Bradley-Terry loss:
$$\nabla_w \mathcal{L} = -\tfrac{1}{N}\textstyle\sum (1-\sigma(z))(\phi_w - \phi_l),\qquad z = (\phi_w - \phi_l)\cdot w$$
Return `w` and the loss history. The tests check held-out accuracy (fraction of pairs where the RM prefers the human's choice) is close to the best achievable, because the labels themselves are noisy.

In [ ]:
def train_rm(Phi_w, Phi_l, steps=500, lr=0.5):
    w = np.zeros(Phi_w.shape[1]); hist = []
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_train_rm(f):
    w, hist = f(train_w, train_l)
    assert hist[-1] < hist[0], "loss must decrease"
    acc = ((test_w - test_l) @ w > 0).mean()
    oracle = ((test_w - test_l) @ w_true > 0).mean()
    assert acc > oracle - 0.03, f"held-out accuracy {acc:.3f} should be close to the oracle's {oracle:.3f}"

check("train_rm", _test_train_rm, train_rm)

## Task 3: From a K-way ranking to pairs

InstructGPT labelers ranked between **K = 4 and K = 9** responses per prompt. A ranking of K items yields $\binom{K}{2}$ comparisons. Given a ranking (best first), return all `(winner, loser)` pairs.
(InstructGPT trains on all $\binom{K}{2}$ pairs from one prompt as a single batch element, because the pairs are highly correlated and treating them as independent caused overfitting.)

In [ ]:
def ranking_to_pairs(ranking):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_ranking_to_pairs(f):
    p = f(['a', 'b', 'c'])
    assert sorted(p) == [('a', 'b'), ('a', 'c'), ('b', 'c')]
    assert len(f(list(range(4)))) == 6 and len(f(list(range(9)))) == 36, "C(4,2)=6, C(9,2)=36"

check("ranking_to_pairs", _test_ranking_to_pairs, ranking_to_pairs)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The pairwise loss
def bt_loss_solution(r_w, r_l):
    return -log_sigmoid(r_w - r_l).mean()

check("bt_loss (reference)", _test_bt_loss, bt_loss_solution)

In [ ]:
# @title Solution for Task 2: Fit a linear reward model
def train_rm_solution(Phi_w, Phi_l, steps=500, lr=0.5):
    w = np.zeros(Phi_w.shape[1]); D = Phi_w - Phi_l; hist = []
    for _ in range(steps):
        z = D @ w
        hist.append(-log_sigmoid(z).mean())
        grad = -((1 - 1 / (1 + np.exp(-z)))[:, None] * D).mean(0)
        w -= lr * grad
    return w, hist

check("train_rm (reference)", _test_train_rm, train_rm_solution)

In [ ]:
# @title Solution for Task 3: From a K-way ranking to pairs
def ranking_to_pairs_solution(ranking):
    return [(ranking[i], ranking[j]) for i in range(len(ranking)) for j in range(i + 1, len(ranking))]

check("ranking_to_pairs (reference)", _test_ranking_to_pairs, ranking_to_pairs_solution)

---
## Experiments
**Accuracy vs data, and the noise ceiling.** Human labels are noisy, so even the true reward function can't predict every choice. Real reward models reach only modest held-out agreement with labelers, which is expected, not a bug. See how accuracy grows with the number of comparisons.

In [ ]:
oracle = ((test_w - test_l) @ w_true > 0).mean()
for n in [25, 100, 400, 2000]:
    w, _ = train_rm_solution(train_w[:n], train_l[:n])
    print(f"{n:5d} comparisons: held-out accuracy {((test_w - test_l) @ w > 0).mean():.3f}   (oracle ceiling {oracle:.3f})")
print("\nLlama 2 trained its reward models on over 1 million human binary comparisons (1,418,091 Meta comparisons).")

---
## Takeaways
* A reward model turns comparisons into a scalar score via the Bradley-Terry loss $-\log\sigma(r_w - r_l)$. Only differences between rewards are identified.
* Comparisons are easier and more consistent for humans than absolute scores, and a K-way ranking gives $\binom{K}{2}$ pairs.
* Accuracy is capped by label noise. A perfect reward model doesn't exist, which is one reason RL against it needs a leash (decks 05–06).